In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run /Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/00.Configurations

In [0]:
%run "/Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/003.gold helper functions"

In [0]:
from pyspark.sql.functions import col,lit

In [0]:
silver_table_constructors = f"{catalog}.{silver_schema}.constructors"
gold_table_nationality = f'{catalog}.{gold_schema}.ref_nationality_region'
gold_table = f"{catalog}.{gold_schema}.dim_constructors"

In [0]:
dim_constructors_df = (
                       spark.table(silver_table_constructors).alias("c").filter(col("batch_id") == lit(v_batch_id))
                      .join(
                          spark.table(gold_table_nationality).alias("r")
                          ,"nationality","left")
                      .select(col("c.constructor_id"),col("c.constructor_name"),col("c.nationality"),col("r.region").alias("nationality_region"))
)

In [0]:
write_to_gold(
    input_df=dim_constructors_df,
    table_name=gold_table,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=["constructor_name","nationality","nationality_region"]
)